In [2]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import col

from pyspark.ml.feature import (
    ImputerModel,
    StringIndexerModel,
    OneHotEncoderModel,
    VectorAssembler,
    StandardScalerModel, 
)

from pyspark.ml.regression import RandomForestRegressionModel

In [3]:
import time

In [4]:
spark = SparkSession.builder \
    .appName("Stock Inference") \
    .getOrCreate()

26/06/26 10:09:27 WARN Utils: Your hostname, Ridhimalaptop resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/26 10:09:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 10:09:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df = spark.read.csv(
    "../data/big_stock_dataset_500k.csv",
    header=True,
    inferSchema=True
)

print(df.count())
print(len(df.columns))

500000
103


In [6]:
drop_cols = [
    "Name_rat",
    "NSE Code_rat",
    "Industry_rat",
    "Days Receivable Outstanding",
    "signal",
    "t_1_price",
    "Credit rating",
    "BSE Code_rat",
    "Current Price_rat",
    "Market Capitalization_rat"
]

existing = [c for c in drop_cols if c in df.columns]

df = df.drop(*existing)

In [7]:
df = (
    df
    .withColumnRenamed("Industry_bal", "Industry")
    .withColumnRenamed("Current Price_bal", "Current Price")
    .withColumnRenamed("Market Capitalization_bal", "Market Capitalization")
)

In [8]:
df = df.filter(col("future_return").isNotNull())

In [9]:
imputer_model = ImputerModel.load(
    "../models/imputer_model"
)

indexer_model = StringIndexerModel.load(
    "../models/indexer_model"
)

encoder_model = OneHotEncoderModel.load(
    "../models/encoder_model"
)

assembler = VectorAssembler.load(
    "../models/vector_assembler"
)

scaler_model = StandardScalerModel.load(
    "../models/scaler_model"
)

rf_model = RandomForestRegressionModel.load(
    "../models/spark_rf_model"
)

In [10]:
start = time.time()

In [11]:
df = imputer_model.transform(df)

df = indexer_model.transform(df)

df = encoder_model.transform(df)

df = assembler.transform(df)

df = scaler_model.transform(df)

26/06/26 10:10:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [12]:
df.count()

print(f"Preprocessing Time : {time.time()-start:.2f} sec")

Preprocessing Time : 4.32 sec


In [13]:
predictions = rf_model.transform(df)

In [14]:
predictions.select(
    "future_return",
    "prediction"
).show(20, truncate=False)

+---------------------+---------------------+
|future_return        |prediction           |
+---------------------+---------------------+
|0.014314561680807166 |-0.0230909992690245  |
|-0.017542442268237805|0.023303840620239553 |
|0.08529645145853225  |0.02903062120003409  |
|0.10823242094043073  |0.06011594487660801  |
|-0.13465579783743287 |-0.029931869929184465|
|-0.018507980476362592|0.017576305482829638 |
|0.028180575815452042 |-0.028322075368467696|
|-0.12177937504869762 |-0.002521270741779187|
|0.07751491131367728  |-0.00454478627261328 |
|-0.04324905507364279 |-0.024333990915860762|
|-0.12978541870786084 |-0.032589847742868376|
|-0.11570670826097193 |-0.008803684960309595|
|0.013839020958732047 |0.04683199988354641  |
|0.10172117523228949  |0.013955422253786433 |
|0.012732127844779735 |-0.01719560747723797 |
|-0.10433634662654145 |-0.04675704824891775 |
|-0.057406523691968414|-0.03209287033438689 |
|-0.12035997667528513 |-0.031874186148840786|
|0.10226162006438243  |0.009157736

In [15]:
start = time.time()
predictions.select(
    "prediction"
).write.mode("overwrite").csv(
    "../outputs/predictions"
)
print(f"Write Time : {time.time()-start:.2f} sec")

Write Time : 73.29 sec


In [16]:
print(predictions.count())

499895


In [17]:
import time

In [18]:
start = time.time()

bdf = spark.read.csv(
    "../data/big_stock_dataset_500k.csv",
    header=True,
    inferSchema=True
)

bdf.count()

print(f"CSV Read Time : {time.time()-start:.2f} sec")

CSV Read Time : 13.60 sec


In [19]:
start = time.time()

predictions = rf_model.transform(df)

predictions.count()

print(f"Inference Time : {time.time()-start:.2f} sec")

Inference Time : 4.22 sec


In [20]:
start = time.time()

predictions = rf_model.transform(df)

predictions.explain(True)

predictions.count()

print(f"Prediction after cache : {time.time()-start:.2f} sec")

== Parsed Logical Plan ==
'Project [Name_bal#17, BSE Code_bal#18, NSE Code_bal#19, Industry#425, Current Price#1025, Debt#1026, Equity capital#1027, Preference capital#1028, Reserves#1029, Secured loan#1030, Unsecured loan#1031, Balance sheet total#1032, Gross block#1033, Revaluation reserve#1034, Accumulated depreciation#1035, Net block#1036, Capital work in progress#1037, Investments#1038, Current assets#1039, Current liabilities#1040, Book value of unquoted investments#1041, Market value of quoted investments#1042, Contingent liabilities#1043, Total Assets#1044, ... 74 more fields]
+- Project [Name_bal#17, BSE Code_bal#18, NSE Code_bal#19, Industry#425, Current Price#1025, Debt#1026, Equity capital#1027, Preference capital#1028, Reserves#1029, Secured loan#1030, Unsecured loan#1031, Balance sheet total#1032, Gross block#1033, Revaluation reserve#1034, Accumulated depreciation#1035, Net block#1036, Capital work in progress#1037, Investments#1038, Current assets#1039, Current liabilit

Prediction after cache : 3.94 sec
